<a href="https://colab.research.google.com/github/ssmartin-code/Espacio_Colap/blob/main/InformesZAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install docxtpl pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os
from docxtpl import DocxTemplate, InlineImage
from docx.shared import Mm
from google.colab import drive

# 1. CONEXIÓN Y RUTAS
drive.mount('/content/drive')

ruta_base = '/content/drive/MyDrive/InformesZAR/'
ruta_excel = os.path.join(ruta_base, 'Info_Ayto.xlsx')
ruta_plantilla = os.path.join(ruta_base, 'Plantilla_ZonasZar.docx')
ruta_escudos = os.path.join(ruta_base, 'Escudos_Aytos/')
ruta_salida = os.path.join(ruta_base, 'Informes_Generados/')

if not os.path.exists(ruta_salida): os.makedirs(ruta_salida)

# =========================================================
# 2. CONFIGURACIÓN DE CAMPOS (FÁCIL DE EDITAR)
# =========================================================
# Estructura:
# 'ETIQUETA_EN_WORD': ('Columna_Exacta_Excel', 'tipo', 'extra_o_tamaño')

CONFIG_CAMPOS = {
    # TEXTOS
    'NombreConcejo':                  ('NombreConcejo', 'texto', None),
    'ComarcaForestal_Perteneciente':  ('ComarcaForestal_Perteneciente', 'texto', None),
    'NombreAlcalde':                 ('Nom_Alcalde', 'texto', None),

    # NÚMEROS (se les pone el punto de miles automáticamente)
    'Superficie_Ha':                  ('Superficie_Concejo_Ha', 'numero', None),
    'Poblacion':                      ('Poblacion_Inventada', 'numero', None),
    'numero_parroquias':              ('N_Parroquias', 'numero', None),

    # IMÁGENES (el último número es el ancho en milímetros)
    'image_escudo':                   ('NombreImagenEscudo', 'imagen', 50),
}
# =========================================================

# 3. PROCESAMIENTO
df = pd.read_excel(ruta_excel)
print(f"Iniciando generación de {len(df)} informes...")

for _, fila in df.iterrows():
    doc = DocxTemplate(ruta_plantilla)
    contexto = {}

    for etiqueta_word, (columna_excel, tipo, extra) in CONFIG_CAMPOS.items():
        try:
            valor_raw = fila[columna_excel]

            if tipo == 'texto':
                contexto[etiqueta_word] = valor_raw

            elif tipo == 'numero':
                # Formatear con puntos de miles: 1000 -> 1.000
                contexto[etiqueta_word] = f"{valor_raw:,}".replace(",", ".")

            elif tipo == 'imagen':
                ruta_img = os.path.join(ruta_escudos, str(valor_raw))
                if os.path.exists(ruta_img):
                    contexto[etiqueta_word] = InlineImage(doc, ruta_img, width=Mm(extra))
                else:
                    print(f"⚠️ Imagen no encontrada para {fila['NombreConcejo']}: {valor_raw}")
                    contexto[etiqueta_word] = "[(Imagen no encontrada)]"

        except KeyError:
            print(f"❌ ERROR: La columna '{columna_excel}' no existe en el Excel.")
            contexto[etiqueta_word] = "ERR: Columna no encontrada"

    # Renderizar y Guardar
    doc.render(contexto)
    nombre_final = f"Informe_Zar_{fila['NombreConcejo']}.docx"
    doc.save(os.path.join(ruta_salida, nombre_final))
    print(f"✅ Generado: {nombre_final}")

print("\n¡Todo listo! Revisa la carpeta en Drive.")

ModuleNotFoundError: No module named 'docxtpl'